In [4]:
import pandas as pd
import random

raw_data = pd.read_csv("nafdac_drugs.csv")
clean_data = raw_data.dropna(subset=['product_name', 'nafdac']).copy()

clean_data['product_name'] = clean_data['product_name'].astype(str).str.strip()
clean_data['nafdac'] = clean_data['nafdac'].astype(str).str.strip()

REAL_NAFDAC_NUMBERS = set(clean_data['nafdac'].unique())

dataset_rows = []
for _, row in clean_data.iterrows():
    genuine_text = f"{row['product_name']} {row['nafdac']}"
    dataset_rows.append({"Text": genuine_text, "Label": 0})

    fake_number = f"FAKE-{random.randint(1000, 9999)}"
    suspicious_text = f"{row['product_name']} {fake_number}"
    dataset_rows.append({"Text": suspicious_text, "Label": 1})

df_train = pd.DataFrame(dataset_rows)
print(f"Dataset prepared with {len(df_train)} total rows.")


Dataset prepared with 18110 total rows.


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

X = df_train["Text"]
y = df_train["Label"]

vectorizer = TfidfVectorizer()
X_vectors = vectorizer.fit_transform(X)

model = MultinomialNB()
model.fit(X_vectors, y)

print("Machine learning model trained successfully on 18,110 records!")


Machine learning model trained successfully on 18,110 records!


In [20]:
def check_my_drug(test_string):
    transformed_input = vectorizer.transform([test_string])
    ml_prediction = model.predict(transformed_input)

    has_exact_registered_nrn = any(nrn in test_string for nrn in REAL_NAFDAC_NUMBERS)

    print(f"🔍 Input: '{test_string}'")
    if ml_prediction == 0 and has_exact_registered_nrn:
        print("💡 OUTPUT: Genuine\n" + "-"*50)
    else:
        print("🚨 OUTPUT: Suspicious\n" + "-"*50)

print("🚀 RUNNING REGISTRY BATCH CHECK:\n" + "="*50)

# Batch test runs using the real file's product name layout constraints
check_my_drug("#Accu-Chek A3-100882")
check_my_drug("Fidson Healthcare Amoxicillin 250mg ERROR-23")
check_my_drug("#Afya 03-1313")
check_my_drug("Juhel Nigeria Vitamin C 100mg INVALID-CODE-99")
check_my_drug("#Always 3 Mar-13")
check_my_drug(" Artemether and Lumefantrine Tablets 80/480   A4-101769")


🚀 RUNNING REGISTRY BATCH CHECK:
🔍 Input: '#Accu-Chek A3-100882'
💡 OUTPUT: Genuine
--------------------------------------------------
🔍 Input: 'Fidson Healthcare Amoxicillin 250mg ERROR-23'
🚨 OUTPUT: Suspicious
--------------------------------------------------
🔍 Input: '#Afya 03-1313'
💡 OUTPUT: Genuine
--------------------------------------------------
🔍 Input: 'Juhel Nigeria Vitamin C 100mg INVALID-CODE-99'
🚨 OUTPUT: Suspicious
--------------------------------------------------
🔍 Input: '#Always 3 Mar-13'
🚨 OUTPUT: Suspicious
--------------------------------------------------
🔍 Input: ' Artemether and Lumefantrine Tablets 80/480   A4-101769'
🚨 OUTPUT: Suspicious
--------------------------------------------------
